In [24]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_community.utilities import SQLDatabase
import pandasai as pai
from pandasai_litellm.litellm import LiteLLM
from litellm import completion
import pandas as pd
import time
import re
import sys
import os
os.environ["PYTHONIOENCODING"] = "utf-8"

In [25]:
df_actor = pd.read_csv("actor.csv", encoding='latin-1')
df_characters = pd.read_csv("characters.csv")
df_movie = pd.read_csv("movie.csv")

In [9]:
def generate_Response(question: str, dataframes: list, model_id: str = None, api_key: str = None):
    """
    Generate a response using PandasAI for the given question.
    
    Args:
        question: The question to ask
        dataframes: List of PandasAI DataFrames to query
        model_id: The LLM model identifier
        api_key: API key for the LLM
    
    Returns:
        tuple: (response, elapsed_time)
    """
    default_api_key = "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"
    default_model = "nvidia_nim/meta/llama-3.1-405b-instruct"
    
    # Configure the LLM
    llm = LiteLLM(
        model=model_id or default_model,
        api_key=api_key or default_api_key,
        stream=False,
    )
    pai.config.set({"llm": llm, "save_charts": False,"verbose": False})
    
    start_time = time.time()
    response = pai.chat(question, *dataframes)
    elapsed_time = time.time() - start_time
    
    return response, elapsed_time

In [17]:
def extract_Generated_code(log_path: str = "pandasai.log"):
    """
    Extract the full generated code from PandasAI log.
    
    Returns:
        dict: {"sql_query": str, "full_code": str}
    """
    from pathlib import Path
    
    log_file = Path(log_path)
    if not log_file.exists():
        return {"sql_query": "", "full_code": ""}
    
    text = log_file.read_text(encoding="utf-8", errors="replace")
    
    # --- Extract full generated code from "Executing code:" section ---
    full_code = ""
    # Pattern to find code after "Executing code:" until next log entry or end
    exec_pattern = r'\[INFO\] Executing code:\s*(.+?)(?=\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} \[INFO\]|$)'
    exec_matches = re.findall(exec_pattern, text, re.DOTALL)
    if exec_matches:
        full_code = exec_matches[-1].strip()
    
    # --- Extract SQL query from the full code ---
    sql_query = ""
    # Pattern for triple-quoted multi-line SQL
    multi_line_pattern = r'sql_query\s*=\s*"""(.+?)"""'
    single_quote_pattern = r"sql_query\s*=\s*'([^']+)'"
    double_quote_pattern = r'sql_query\s*=\s*"([^"]+)"'

    # Search in full_code first, then in entire text
    search_text = full_code if full_code else text
    
    matches = re.findall(multi_line_pattern, search_text, re.DOTALL)
    if not matches:
        matches = re.findall(single_quote_pattern, search_text)
    if not matches:
        matches = re.findall(double_quote_pattern, search_text)
    if matches:
        sql_query = matches[-1].strip()
    
    return {"sql_query": sql_query, "full_code": full_code}


def clear_log(log_path: str = "pandasai.log"):
    """Clear the PandasAI log file after extraction."""
    from pathlib import Path
    log_file = Path(log_path)
    if log_file.exists():
        log_file.write_text("", encoding="utf-8")

In [5]:
def validation_Response(query: str, question: str,response=None,answer=None) -> bool:
    if not query:
        return False
    
    try:
        validation = completion(
            model="nvidia_nim/meta/llama-3.1-405b-instruct",
            messages=[{
                "role": "user", 
                "content": f"""You are a query validator.
                Question: {question}
                Code: {query}
                response: {response} if response is chart or visualization, compare answer with the query else compare response with the answer
                answer: {answer}
                Does this code correctly retrieve the data required to answer the question?
                Respond with ONLY one word: TRUE or FALSE."""
            }],
            api_key= "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"
            
        )
        return validation.choices[0].message.content.strip()
        
    except Exception as e:
        print(f"⚠️ Validation error: {e}")
        return None

In [33]:
clear_log()
# Simple test using the helper functions
question = "List the character's name of actress born in Sherman Oaks and starred in the movie Bruce Almighty with height greater than the 50% of average height of all actors listed."

# Use generate_Response function
response, elapsed_time = generate_Response(question, [df_actor, df_characters, df_movie])
print(f"Response: {response.value if hasattr(response, 'value') else response}")

# Extract full generated code
generated = extract_Generated_code()
print(f"\nSQL Query:\n{generated['sql_query']}")
print(f"\nFull Code:\n{generated['full_code']}")

# Use validation_Response function with full_code for complete validation
code_to_validate = generated['full_code'] if generated['full_code'] else generated['sql_query']
is_correct = validation_Response(code_to_validate, question, response=response,answer="Elizabeth Olsen")
print(f"\nIs Correct: {is_correct}")

# Check if validation returned "FALSE" (string comparison, case-insensitive)
# if is_correct and "false" in str(is_correct).lower():
#     explanation = error_explaination_Response(code_to_validate, question, response)
#     print(f"\nError Explanation:\n{explanation}")

print(f"Time taken: {elapsed_time:.2f} seconds")

Response:    Character Name
0  Grace Connelly

SQL Query:
SELECT T2."Character Name"
FROM table_c904f18044a81521434d5989d20e4d77 AS T1
JOIN table_023dffb429e6a0d269eb875dbfdb3c52 AS T2
ON T1."﻿ActorID" = T2.ActorID
JOIN table_8740782e3632f06a100a9ce037f8a8c8 AS T3
ON T2.MovieID = T3.MovieID
WHERE T3.Title = 'Bruce Almighty'
AND T1."Birth City" = 'Sherman Oaks'
AND T1."Height (Inches)" > (SELECT AVG("Height (Inches)") * 0.5 FROM table_c904f18044a81521434d5989d20e4d77)

Full Code:
import pandas as pd
sql_query = """
SELECT T2."Character Name"
FROM table_c904f18044a81521434d5989d20e4d77 AS T1
JOIN table_023dffb429e6a0d269eb875dbfdb3c52 AS T2
ON T1."﻿ActorID" = T2.ActorID
JOIN table_8740782e3632f06a100a9ce037f8a8c8 AS T3
ON T2.MovieID = T3.MovieID
WHERE T3.Title = 'Bruce Almighty'
AND T1."Birth City" = 'Sherman Oaks'
AND T1."Height (Inches)" > (SELECT AVG("Height (Inches)") * 0.5 FROM table_c904f18044a81521434d5989d20e4d77)
"""
result_df = execute_sql_query(sql_query)
result = {'type': 'da

In [26]:
DB_PATH = "movie.sqlite"         
MODEL = "meta/llama-3.1-405b-instruct"   
NVIDIA_API_KEY = "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"

In [27]:
db = SQLDatabase.from_uri(
    f"sqlite:///{DB_PATH}",
    include_tables=["actor", "movie", "characters"],
    sample_rows_in_table_info=3,          # shows NIM 3 sample rows for context
)
print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print("✅ Connected to database")
print("Tables:", db.get_usable_table_names())

Dialect: sqlite
Available tables: ['actor', 'characters', 'movie']
✅ Connected to database
Tables: ['actor', 'characters', 'movie']


In [28]:
llm = ChatNVIDIA(
    model=MODEL,
    nvidia_api_key=NVIDIA_API_KEY,
  
)

In [29]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [30]:
system_prompt = """
You are an agent designed to interact with a SQL database.
Given an input question, create a syntactically correct {dialect} query to run,
then look at the results of the query and return the answer. Unless the user
specifies a specific number of examples they wish to obtain, always limit your
query to at most {top_k} results.

You can order the results by a relevant column to return the most interesting
examples in the database. Never query for all the columns from a specific table,
only ask for the relevant columns given the question.

You MUST double check your query before executing it. If you get an error while
executing a query, rewrite the query and try again.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the
database.

To start you should ALWAYS look at the tables in the database to see what you
can query. Do NOT skip this step.

Then you should query the schema of the most relevant tables.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [31]:
from langchain.agents import create_agent


agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
)

In [32]:
question = "Which movie had the biggest budget? Give the name of the movie."

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which movie had the biggest budget? Give the name of the movie.
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (chatcmpl-tool-baebbac50cca9fe4)
 Call ID: chatcmpl-tool-baebbac50cca9fe4
  Args:
    tool_input:
================================= Tool Message =================================
Name: sql_db_list_tables

actor, characters, movie
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (chatcmpl-tool-9b58f072c99d3784)
 Call ID: chatcmpl-tool-9b58f072c99d3784
  Args:
    table_names: movie
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE movie (
	"MovieID" INTEGER, 
	"Title" TEXT, 
	"MPAA Rating" TEXT, 
	"Budget" INTEGER, 
	"Gross" INTEGER, 
	"Release Date" TEXT, 
	"Genre" TEXT, 
	"Runtime" INTEGER, 
	"Rating" REAL